# Recursive Character Splitting

**Recursive Character Splitting** is a text chunking technique commonly used in **RAG applications**.

The goal is to split a large document into smaller chunks while trying to keep related text together.

## How it works

You provide:

- **Chunk size** → Maximum size of each chunk
- **Chunk overlap** → How much text should be shared between consecutive chunks
- **Separators** → Characters used to split the text

A common separator order is:

```text
[
    "\n\n",  # Paragraph
    "\n",    # Line
    " ",     # Word
    ""       # Character
]
```

The splitter works **recursively**.

It tries the first separator. If the resulting chunk is still too large, it tries the next separator.

---

# Example 1: Split by Paragraph

### Original Text

```text
Python is a popular programming language. It is easy to learn.

Python is used for web development, AI, automation, and data analysis.
```

Assume:

```text
Chunk Size = 80 characters
Separators = ["\n\n", "\n", " ", ""]
```

### Step 1: Try `\n\n`

The text contains two paragraphs.

```text
Paragraph 1:
Python is a popular programming language. It is easy to learn.

Paragraph 2:
Python is used for web development, AI, automation, and data analysis.
```

Both paragraphs are smaller than the chunk size.

### Result

```text
Chunk 1:
Python is a popular programming language. It is easy to learn.

Chunk 2:
Python is used for web development, AI, automation, and data analysis.
```

Here, the splitter **does not need to try** `\n`, `" "`, or `""`.

---

# Example 2: Paragraph Is Too Large → Split by Line

### Original Text

```text
Python is a popular programming language used by developers worldwide.
It supports object-oriented programming.
It is commonly used for AI and data science.
It can also be used for web development.
```

Assume:

```text
Chunk Size = 100 characters

Separators = [
    "\n\n",
    "\n",
    " ",
    ""
]
```

### Step 1: Try `\n\n`

There is no paragraph separator.

So it moves to:

```text
"\n"
```

### Step 2: Split by New Line

```text
Chunk 1:
Python is a popular programming language used by developers worldwide.
It supports object-oriented programming.

Chunk 2:
It is commonly used for AI and data science.
It can also be used for web development.
```

The text is split using line boundaries.

---

# Example 3: Line Is Too Large → Split by Word

### Original Text

```text
Artificial intelligence is transforming healthcare by helping doctors analyze medical images detect diseases and make faster decisions for patients.
```

Assume:

```text
Chunk Size = 60 characters

Separators = [
    "\n\n",
    "\n",
    " ",
    ""
]
```

### Step 1: Try Paragraph

```text
"\n\n"
```

No paragraph separator.

### Step 2: Try New Line

```text
"\n"
```

No new line.

### Step 3: Try Space

Now the splitter separates the text using spaces and creates chunks close to the maximum size.

```text
Chunk 1:
Artificial intelligence is transforming healthcare by helping

Chunk 2:
doctors analyze medical images detect diseases and make

Chunk 3:
faster decisions for patients.
```

This is better than cutting randomly because it tries to keep **complete words together**.

---

# Example 4: A Single Word Is Too Large → Split by Character

Sometimes there may be a very long string with no spaces.

### Original Text

```text
ThisIsAnExtremelyLongWordWithoutAnySpacesOrSeparatorsThatNeedsToBeSplit
```

Assume:

```text
Chunk Size = 20

Separators = [
    "\n\n",
    "\n",
    " ",
    ""
]
```

### Step 1

Try:

```text
"\n\n"
```

❌ No match.

### Step 2

Try:

```text
"\n"
```

❌ No match.

### Step 3

Try:

```text
" "
```

❌ No spaces.

### Step 4: Split by Character

```text
Chunk 1:
ThisIsAnExtremelyLo

Chunk 2:
ngWordWithoutAnySpa

Chunk 3:
cesOrSeparatorsThat

Chunk 4:
NeedsToBeSplit
```

The final separator:

```text
""
```

means **split character by character if necessary**.

---

# Visual Flow

```text
                    Large Document
                          │
                          ▼
                 Try "\n\n" (Paragraph)
                          │
              ┌───────────┴───────────┐
              │                       │
         Small enough?               Too large
              │                       │
             YES                      ▼
              │                  Try "\n"
              ▼                   (Line)
         Create Chunk                  │
                              ┌────────┴────────┐
                              │                 │
                         Small enough?       Too large
                              │                 │
                             YES                ▼
                              │             Try " "
                              ▼             (Word)
                         Create Chunk            │
                                          Too large?
                                               │
                                               ▼
                                           Try ""
                                        (Character)
```

# Simple Real-Life Example

Imagine you have a document:

```text
Document
│
├── Paragraph
│   ├── Sentence
│   │   ├── Words
│   │   │   └── Characters
```

Recursive Character Splitting tries to preserve the **largest meaningful structure first**:

```text
1. Paragraph
       ↓
2. Line
       ↓
3. Word
       ↓
4. Character
```

It only goes to the next level when the current text is **too large for the configured chunk size**.

# Why Is It Useful for RAG?

Suppose you have a PDF with:

```text
Chapter
    ↓
Paragraph
    ↓
Sentence
    ↓
Words
```

Instead of randomly cutting text like this:

```text
Chunk 1:
...machine learning is used in health

Chunk 2:
care to identify diseases...
```

Recursive Character Splitting tries to preserve natural boundaries:

```text
Chunk 1:
Machine learning is used in healthcare to identify diseases.

Chunk 2:
Deep learning models can analyze medical images.
```

This usually produces better chunks for:

- **Embeddings**
- **Vector databases**
- **Similarity search**
- **RAG pipelines**
- **LLM context retrieval**

## In one sentence

> **Recursive Character Splitting starts with the largest separator, such as paragraphs, and recursively moves to smaller separators like lines, words, and finally characters until the text fits within the desired chunk size.**
